# MODUL PRAKTIKUM BIG DATA
## Pertemuan 10 — Pemrosesan Data Streaming dengan Spark Structured Streaming (Arch Linux Safe Edition)

| Informasi | Keterangan |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Dosen Pengampu** | Alifia Revan Prananda, S.Pd., M.Eng. |
| **Sistem Operasi** | Arch Linux / CachyOS (Userspace Zero-Conflict Environment) |
| **Praktikan** | Sholahuddin Ahmad (NPM: 250506070) |
| **Software Stack** | Apache Spark 3.5.9, OpenJDK 11, uv (Python 3.11.16) |
| **Topik** | Batch vs Streaming, Spark Structured Streaming, Window Agregasi, File Sink |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-9 selesai |

---

### 🛡️ Jaminan Keamanan Sistem & Petunjuk Penggunaan (Arch Linux)
1. **Tidak Ada Instalasi Baru**: Spark Structured Streaming sudah merupakan bagian built-in dari runtime Apache Spark 3.5.9.
2. **Java 11 Terisolasi Penuh**: Eksekusi Spark dikunci pada OpenJDK 11 (`/usr/lib/jvm/java-11-openjdk`) melalui `spark_init.py` agar tidak bentrok dengan Java bawaan host Arch Linux.
3. **Optimasi Partisi Shuffle**: Konfigurasi `spark.conf.set("spark.sql.shuffle.partitions", "4")` diterapkan untuk performa optimal pada laptop single-node.

---
## Recap Pertemuan Sebelumnya & Konsep Inti

- [ ] Memahami `SparkSession`, DataFrame, `groupBy`/`agg` dari Pertemuan 4-7
- [ ] Memahami ekosistem NoSQL MongoDB dari Pertemuan 9

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 10, mahasiswa mampu:
1. Menjelaskan perbedaan mendasar antara pemrosesan **batch** dan **streaming**.
2. Menjelaskan konsep dasar Spark Structured Streaming: *input source*, *trigger*, *output sink*, *output mode*.
3. Membangun sebuah *streaming query* yang membaca data secara inkremental dan melakukan agregasi real-time.
4. Menerapkan agregasi berbasis **window waktu** dan penulisan streaming ke **berkas Parquet**.

---

## 10.1 Batch vs Streaming: Mengapa Micro-Batch?

Sepanjang Pertemuan 3-9, seluruh data yang kita olah bersifat **statis** (batch processing). Di industri nyata, data seringkali **terus mengalir tanpa henti** (stream processing) seperti transaksi e-commerce, sensor IoT, atau log klik pengguna.

| Aspek | Batch Processing | Stream Processing |
|---|---|---|
| **Sifat data** | Statis, sudah lengkap tersimpan | Terus mengalir / bertambah (unbounded) |
| **Kapan diproses** | Terjadwal berkala (mis. harian) | Seketika saat data tiba (near real-time) |
| **Contoh nyata** | Rekap keuangan bulanan | Deteksi penipuan transaksi (fraud detection) |

**Spark Structured Streaming** memungkinkan kita menggunakan DataFrame API yang sama persis dengan batch, di mana Spark menerjemahkannya menjadi rangkaian pemrosesan *micro-batch* berkecepatan tinggi.

---
## 10.2 Persiapan: SparkSession & Folder Simulasi Streaming

In [ ]:
from spark_init import spark, F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Mengurangi partisi shuffle default (200) ke 4 agar komputasi lokal ringan & cepat
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"⚡ SparkSession aktif. Folder streaming '{folder_stream}' siap.")

> **Wajib Mendefinisikan Skema Eksplisit:** Berbeda dengan batch (`inferSchema=True`), streaming tidak dapat menebak skema secara otomatis karena data bersifat terbuka (*unbounded*). Skema harus didefinisikan di awal dengan `StructType`.

In [ ]:
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema transaksi terdefinisi:")
print(skema_transaksi)

---
## 10.3 Membangun Streaming Query Pertama

### Tahap 1 & 2: Input Source (`readStream`) dan Transformasi

In [ ]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

agregasi_stream = df_stream.groupBy("kota").agg(
    F.sum("pendapatan").alias("total_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

print("df_stream.isStreaming:", df_stream.isStreaming)

### Tahap 3: Output Sink (`writeStream`) dengan Memory Sink

In [ ]:
query1 = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_agregasi") \
    .start()

print("Streaming query dimulai. Status:", query1.status)

### Tahap 4: Mengalirkan Batch Data & Mengamati Akumulasi State

In [ ]:
# Batch 1 tiba
pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kota": ["Magelang", "Solo", "Magelang"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [150000, 200000, 75000],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

time.sleep(6)
print("=== HASIL AGREGASI SETELAH BATCH 1 ===")
spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

# Batch 2 tiba
pd.DataFrame({
    "trx_id": ["T4", "T5"],
    "kota": ["Magelang", "Yogyakarta"],
    "kategori": ["Fashion", "Elektronik"],
    "pendapatan": [300000, 500000],
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

time.sleep(6)
print("=== HASIL AGREGASI SETELAH BATCH 2 (STATE TERAKUMULASI) ===")
spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

### Tahap 5: Menghentikan Streaming Query

In [ ]:
query1.stop()
print("Streaming query 1 dihentikan. Status aktif:", query1.isActive)

---
## 10.4 Agregasi Berbasis Window Waktu (Tumbling Window)

In [ ]:
skema_dengan_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

folder_stream2 = "simulasi_stream_waktu"
shutil.rmtree(folder_stream2, ignore_errors=True)
os.makedirs(folder_stream2, exist_ok=True)

df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_dengan_waktu) \
    .option("header", True) \
    .load(folder_stream2)

agregasi_window = df_stream2.groupBy(
    F.window(F.col("waktu_transaksi"), "1 minute")
).agg(
    F.sum("pendapatan").alias("total_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_window") \
    .start()

waktu_sekarang = datetime.now()
pd.DataFrame({
    "trx_id": ["W1", "W2"],
    "kota": ["Semarang", "Solo"],
    "pendapatan": [120000, 180000],
    "waktu_transaksi": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream2}/batch_w1.csv", index=False)

time.sleep(6)
print("=== HASIL AGREGASI WINDOW 1 MENIT ===")
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_transaksi FROM hasil_window").show(truncate=False)

query2.stop()
print("Query 2 dihentikan.")

---
## Latihan Mandiri (Arch Linux Solution)

In [ ]:
folder_latihan = "latihan10_stream"
shutil.rmtree(folder_latihan, ignore_errors=True)
os.makedirs(folder_latihan, exist_ok=True)

skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

df_stream_latihan = spark.readStream \
    .format("csv") \
    .schema(skema_latihan) \
    .option("header", True) \
    .load(folder_latihan)

# Soal 1: Agregasi rata-rata (avg)
agregasi_latihan = df_stream_latihan.groupBy("kategori").agg(
    F.avg("pendapatan").alias("rata_rata_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

query_latihan = agregasi_latihan.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("latihan_agg") \
    .start()

# Soal 2: Batch 1
pd.DataFrame({
    "trx_id": ["L1", "L2", "L3"],
    "kategori": ["Elektronik", "Makanan", "Elektronik"],
    "pendapatan": [200000, 50000, 400000]
}).to_csv(f"{folder_latihan}/batch_latihan_1.csv", index=False)

time.sleep(6)
print("=== HASIL LATIHAN SETELAH BATCH 1 ===")
spark.sql("SELECT * FROM latihan_agg").show()

# Soal 3: Batch 2
pd.DataFrame({
    "trx_id": ["L4", "L5"],
    "kategori": ["Elektronik", "Makanan"],
    "pendapatan": [600000, 150000]
}).to_csv(f"{folder_latihan}/batch_latihan_2.csv", index=False)

time.sleep(6)
print("=== HASIL LATIHAN SETELAH BATCH 2 ===")
spark.sql("SELECT * FROM latihan_agg").show()

query_latihan.stop()
print("Query latihan selesai.")

### Jawaban Refleksi Latihan Soal 4
Spark Structured Streaming mewajibkan pendefinisian skema secara eksplisit (`StructType`) karena data streaming bersifat tidak terbatas (*unbounded*) dan terus mengalir secara inkremental dari waktu ke waktu. Tidak seperti pemrosesan batch yang dapat memindai seluruh berkas terlebih dahulu untuk menyimpulkan tipe kolom (`inferSchema`), sistem streaming tidak memiliki kondisi akhir untuk menunggu kelengkapan data. Definisi skema eksplisit menjamin kepastian kontrak data (*data contract*) sejak micro-batch pertama, mencegah kegagalan pipeline di tengah jalan, dan menjaga konsistensi state akumulasi data terdistribusi.

---
## TUGAS MANDIRI: Dashboard Streaming Kopi Nusantara (Bagian A sampai E)

Pengerjaan kode Tugas Mandiri diorganisasi secara bersih dan terstruktur pada berkas **`tugas.ipynb`**.